# 09 -- Outfield Opportunity and Execution Analysis (Contact Luck v0.7A / v0.7B)

Estimates outfield-opportunity DIFFICULTY (v0.7A: `P(an average MLB outfielder converts
this into an out)`) and DEFENSIVE EXECUTION (v0.7B: actual result vs. that opportunity
-implied expectation), for outfield air balls only. See `mlb_luck_score.models.
compare_opportunity_models` module docstring for the full public-data audit that shaped
this scope, and README.md "Outfield opportunity and execution (Version 0.7A/0.7B)".

**Public-data audit summary**: defender starting location, endpoint, real (measured) hang
time, distance needed, and per-play catch-probability inputs are all NOT available in
public Statcast data or any installed `pybaseball` function (those only expose
season-level aggregate leaderboards). Responsible-fielder identity IS available
(`hit_location` + `fielder_7`/`fielder_8`/`fielder_9`). This notebook therefore evaluates
only `measured_contact_only_v07` -- landing-location estimate, estimated hang time
(physics), wall proximity, exit velocity/launch angle, batted-ball type, and coarse
outfield alignment. `typical_position_proxy_v07` was investigated (citable sources found
were 2015-2016 league averages, judged too stale given a documented trend toward deeper
outfield positioning since then) and is NOT implemented.

Only 2021-2024 data is used here; 2025 remains untouched.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

DATA_PATH = Path("../data/processed/cleaned_development_data_with_geometry.parquet")
df = pd.read_parquet(DATA_PATH) if DATA_PATH.exists() else None
print(f"Loaded {len(df):,} rows" if df is not None else "Data not found -- run make join-park-geometry first")

Loaded 494,173 rows


## 1. Outfield-opportunity eligibility

In [2]:
from mlb_luck_score.eligibility import add_outfield_opportunity_eligibility

elig_df = None
if df is not None:
    elig_df = add_outfield_opportunity_eligibility(df)
    print(elig_df["outfield_opportunity_eligible"].sum(), "eligible rows of", len(elig_df))
    print()
    print(elig_df["outfield_opportunity_exclusion_reason"].value_counts(dropna=False))
    print()
    eligible_only = elig_df[elig_df["outfield_opportunity_eligible"]]
    print("assigned_outfield_position_source:")
    print(eligible_only["assigned_outfield_position_source"].value_counts(dropna=False))

227334 eligible rows of 494173

outfield_opportunity_exclusion_reason
not_outfield_air_ball_bb_type    241374
None                             227334
infield_credited_hit_location     17735
base_training_ineligible           7730
Name: count, dtype: int64

assigned_outfield_position_source:
assigned_outfield_position_source
hit_location          203249
spray_sector_proxy     24085
Name: count, dtype: int64


## 2. Feature engineering: estimated hang time and landing location

Both are physics-derived ESTIMATES (vacuum projectile motion, no drag model) with NO
public ground truth to validate against -- see `mlb_luck_score.data.outfield_physics`.

In [3]:
from mlb_luck_score.features.build_contact_features import add_outfield_opportunity_features

opp_df = None
if elig_df is not None:
    opp_df = add_outfield_opportunity_features(elig_df)
    eligible = opp_df[opp_df["outfield_opportunity_eligible"]]
    print(eligible["estimated_hang_time_s"].describe())
    print()
    print(eligible[["spray_angle_approx", "hit_distance_sc", "landing_x_ft", "landing_y_ft"]].head(5))

count    227334.000000
mean          3.817594
std           1.509076
min           0.100000
25%           2.593424
50%           3.707603
75%           4.926615
max           8.700136
Name: estimated_hang_time_s, dtype: float64

    spray_angle_approx  hit_distance_sc  landing_x_ft  landing_y_ft
1           -30.943301              283   -145.515652    242.722465
4             0.854277              388      5.784843    387.956873
7             9.994220              349     60.568541    343.704018
9           -41.991938              167   -111.727347    124.120909
10          -30.674549              195    -99.481378    167.715400


## 3. Train and evaluate `measured_contact_only_v07`

In [4]:
from mlb_luck_score.models.compare_opportunity_models import (
    VARIANT_MEASURED_CONTACT_ONLY,
    run_opportunity_model_evaluation,
)

comparison = None
trained_models = None
p_out_by_variant = None
if df is not None:
    comparison, trained_models, p_out_by_variant = run_opportunity_model_evaluation(df)
    summary = comparison[VARIANT_MEASURED_CONTACT_ONLY]
    print(f"log_loss={summary['binary_log_loss']:.6f}  ece={summary['expected_calibration_error']:.6f}  "
          f"brier={summary['brier_score']:.6f}  n={summary['sample_count']:,}")
    print(f"converted_to_out_rate={summary['converted_to_out_rate']:.4f}  "
          f"mean_predicted_p_out={summary['mean_predicted_p_out']:.4f}")

log_loss=0.385044  ece=0.017771  brier=0.122296  n=57,598
converted_to_out_rate=0.5462  mean_predicted_p_out=0.5386


## 4. Required-subgroup calibration

In [5]:
if comparison is not None:
    rows = []
    for label, sub in comparison[VARIANT_MEASURED_CONTACT_ONLY]["opportunity_subgroups"].items():
        rows.append({"subgroup": label, "n": sub["sample_count"], "ece": sub["ece"], "log_loss": sub["log_loss"]})
    print(pd.DataFrame(rows).set_index("subgroup"))
    print()
    print("Note:", comparison[VARIANT_MEASURED_CONTACT_ONLY]["movement_direction_subgroup_note"])

                                  n       ece  log_loss
subgroup                                               
bb_type_fly_ball              31650  0.023065  0.346395
bb_type_line_drive            25948  0.045781  0.432187
left_field                    18908  0.031484  0.402773
center_field                  19693  0.027570  0.356635
right_field                   18962  0.020485  0.397564
near_wall_5ft                  1947  0.253951  0.810098
near_wall_10ft                 3983  0.233882  0.764817
near_wall_20ft                 7853  0.180439  0.670934
opportunity_time_q1_shortest  14410  0.046635  0.264809
opportunity_time_q2           14398  0.073916  0.646029
opportunity_time_q3           14390  0.064531  0.427277
opportunity_time_q4_longest   14400  0.034097  0.202210

Note: forward/lateral/backward defender movement direction requires a real defender starting position, which is not available in public Statcast data -- see module docstring's public-data audit. Not computed.


### Interpretation

Calibration is materially worse for near-wall plays (ECE roughly 10x the overall figure)
and for the two middle opportunity-time quartiles -- the model is least reliable exactly
where judging a defensive play is hardest (ambiguous warning-track plays, borderline
opportunities), which is an honest limitation to carry into any v0.7B execution reading
for those specific plays.

## 5. Per-venue calibration

In [6]:
if comparison is not None:
    venue_df = pd.DataFrame(comparison[VARIANT_MEASURED_CONTACT_ONLY]["calibration_by_venue"])
    reliable = venue_df[venue_df["reliable"]].sort_values("ece", ascending=False)
    print(f"{len(reliable)} reliably-sampled venues")
    print(reliable.head(10))

30 reliably-sampled venues
   venue_id  sample_count       ece  reliable
19        3          1895  0.048296      True
7        10          1973  0.045551      True
13        7          1938  0.044948      True
29      680          1736  0.044348      True
25     2395          1846  0.043423      True
23       17          1856  0.042783      True
24     2394          1847  0.041853      True
28     4705          1754  0.039048      True
21     2889          1887  0.036147      True
8      4169          1963  0.033864      True


## 6. Per-defender calibration (reliably-sampled only)

In [7]:
if comparison is not None:
    defender_df = pd.DataFrame(comparison[VARIANT_MEASURED_CONTACT_ONLY]["calibration_by_defender"])
    print(f"{len(defender_df)} defenders with >= min-sample threshold")
    print(defender_df.sort_values("sample_count", ascending=False).head(10))

164 defenders with >= min-sample threshold
   responsible_outfielder_id  sample_count  actual_out_rate  mean_predicted_p_out       ece
0                     686668           707         0.602546              0.581258  0.046338
1                     682998           656         0.545732              0.555280  0.028072
2                     680776           647         0.550232              0.552592  0.052255
3                     696285           623         0.622793              0.597994  0.053282
4                     665750           617         0.607780              0.609852  0.036556
5                     668709           615         0.637398              0.641290  0.036684
6                     621493           602         0.501661              0.457951  0.047453
7                     701538           595         0.647059              0.637017  0.043937
8                     664023           591         0.519459              0.495653  0.056998
9                     665742         

## 7. Game-level bootstrap CI on the model's own metrics

In [8]:
from mlb_luck_score.config import VALIDATION_SEASONS
from mlb_luck_score.features.build_contact_features import OPPORTUNITY_TARGET_COLUMN
from mlb_luck_score.models.compare_opportunity_models import (
    _prepare_opportunity_columns,
    compute_opportunity_bootstrap,
)
from mlb_luck_score.models.compare_park_aware import _prepare_venue_column

bootstrap_result = None
val_df = None
if comparison is not None:
    prepared = _prepare_opportunity_columns(df)
    prepared = _prepare_venue_column(prepared) if "venue_id" in prepared.columns else prepared
    eligible_full = prepared[prepared["outfield_opportunity_eligible"].astype(bool)]
    val_df = eligible_full[eligible_full["season"].isin(VALIDATION_SEASONS)]
    y_true = val_df[OPPORTUNITY_TARGET_COLUMN].astype(int).to_numpy()
    bootstrap_result = compute_opportunity_bootstrap(
        y_true, p_out_by_variant[VARIANT_MEASURED_CONTACT_ONLY], val_df["game_pk"]
    )
    print(json.dumps(bootstrap_result, indent=2))

{
  "log_loss": {
    "point_estimate": 0.38504427546126496,
    "ci_low": 0.38022452456451405,
    "ci_high": 0.3897769859660314,
    "n_reps": 500,
    "seed": 42,
    "resampling_unit": "game_pk"
  },
  "ece": {
    "point_estimate": 0.0177708946670292,
    "ci_low": 0.015366933703491391,
    "ci_high": 0.020986103947931732,
    "n_reps": 500,
    "seed": 42,
    "resampling_unit": "game_pk"
  }
}


## 8. Validation summary

In [9]:
from mlb_luck_score.models.compare_opportunity_models import summarize_opportunity_validation

if comparison is not None:
    validation_summary = summarize_opportunity_validation(
        comparison, {VARIANT_MEASURED_CONTACT_ONLY: bootstrap_result}
    )
    print(json.dumps(validation_summary["per_candidate"], indent=2, default=str))
    print()
    print("typical_position_proxy_v07_built:", validation_summary["typical_position_proxy_v07_built"])
    print(validation_summary["typical_position_proxy_v07_note"])

{
  "measured_contact_only_v07": {
    "overall_ece": 0.0177708946670292,
    "overall_ece_within_threshold": true,
    "subgroup_ece_issues": [
      "near_wall_5ft",
      "near_wall_10ft",
      "near_wall_20ft",
      "opportunity_time_q2",
      "opportunity_time_q3"
    ],
    "venue_ece_issues": [],
    "bootstrap": {
      "log_loss": {
        "point_estimate": 0.38504427546126496,
        "ci_low": 0.38022452456451405,
        "ci_high": 0.3897769859660314,
        "n_reps": 500,
        "seed": 42,
        "resampling_unit": "game_pk"
      },
      "ece": {
        "point_estimate": 0.0177708946670292,
        "ci_low": 0.015366933703491391,
        "ci_high": 0.020986103947931732,
        "n_reps": 500,
        "seed": 42,
        "resampling_unit": "game_pk"
      }
    },
    "passes_basic_validation": false
  }
}

typical_position_proxy_v07_built: False
Not built -- no era-appropriate (2021-2024), citable public source for typical outfielder starting depth was found. Se

## 9. Version 0.7B: defensive execution examples

`defensive_execution = actual_out_indicator - p_out_opportunity`. Positive = the defense
outperformed the opportunity-implied expectation (bad for the batter); negative = the
defense underperformed (good for the batter). `batter_favorable_defensive_circumstance`
is the negated, batter-perspective version -- see `mlb_luck_score.scoring.
defensive_execution` module docstring for the full sign-convention writeup.

In [10]:
from mlb_luck_score.scoring.defensive_execution import compute_defensive_execution

execution_df = None
if val_df is not None:
    p_out = p_out_by_variant[VARIANT_MEASURED_CONTACT_ONLY].reindex(val_df.index)
    actual = val_df[OPPORTUNITY_TARGET_COLUMN].astype(int)
    execution_df = compute_defensive_execution(actual, p_out)
    print(execution_df.describe())
    print()
    print("Strongest POSITIVE defensive execution (converted the toughest opportunities):")
    print(execution_df.nlargest(5, "defensive_execution"))
    print()
    print("Strongest NEGATIVE defensive execution (missed the easiest opportunities):")
    print(execution_df.nsmallest(5, "defensive_execution"))

       defensive_execution  batter_favorable_defensive_circumstance
count         57598.000000                             57598.000000
mean              0.007641                                -0.007641
std               0.349628                                 0.349628
min              -0.999786                                -1.000000
25%              -0.121729                                -0.181819
50%               0.017799                                -0.017799
75%               0.181819                                 0.121729
max               1.000000                                 0.999786

Strongest POSITIVE defensive execution (converted the toughest opportunities):
        defensive_execution  batter_favorable_defensive_circumstance
406266             1.000000                                -1.000000
373548             1.000000                                -1.000000
371284             0.999999                                -0.999999
452845             0.999980     

## 10. All four components together (NOT combined into one score)

For each play: contact expectation (existing Version 0.2 model), opportunity difficulty
(Version 0.7A), defensive execution (Version 0.7B), and residual contact luck (existing
Version 0.2 formula). Reported side by side -- per the task's explicit instruction, these
are NOT summed into a single score here.

In [11]:
from mlb_luck_score.config import TRAIN_SEASONS
from mlb_luck_score.models.train_contact_model import train_model
from mlb_luck_score.scoring.air_ball_components import build_air_ball_component_report

report = None
if val_df is not None:
    eligible_full = prepared[prepared["outfield_opportunity_eligible"].astype(bool)]
    train_df = eligible_full[eligible_full["season"].isin(TRAIN_SEASONS)]
    contact_trained = train_model(train_df, class_weight=None)
    opportunity_trained = trained_models[VARIANT_MEASURED_CONTACT_ONLY]
    report = build_air_ball_component_report(val_df, contact_trained, opportunity_trained)
    print(report.describe())
    print()
    print(report.head(10))

             game_pk  at_bat_number  pitch_number  expected_run_value_contact_model  actual_run_value  residual_contact_luck_runs  p_out_opportunity  \
count        57598.0        57598.0       57598.0                      57598.000000      57598.000000                57598.000000       5.759800e+04   
mean   746012.821192      37.943036      3.358433                          0.207810          0.199701                   -0.008108       5.385937e-01   
std       700.996755      22.234173      1.868264                          0.381775          0.555836                    0.384241       3.502822e-01   
min         744795.0            1.0           1.0                         -0.251891         -0.254916                   -1.537148       3.005768e-11   
25%         745410.0           19.0           2.0                         -0.137023         -0.254916                   -0.174882       1.765743e-01   
50%         746019.0           37.0           3.0                          0.161238     

## 12. Version 0.7C: near-wall opportunity correction

`measured_contact_only_v07`'s calibration is materially worse for near-wall plays (see
Section 4: ECE 0.18-0.25 in the 5/10/20 ft wall bands vs. an overall 0.018). Version 0.7C
builds a GATED architecture (`mlb_luck_score.models.outfield_gating`) to address this
specifically, without touching open-field scoring:

- `open_field_v07` -- the EXISTING `measured_contact_only_v07` model, unchanged, for plays
  NOT within 20 ft of the wall.
- `near_wall_v07c_candidate` -- a specialist trained ONLY on wall-adjacent plays, comparing
  a logistic baseline against an unweighted `HistGradientBoostingClassifier` (can find
  nonlinear interactions among hang time/launch angle/wall distance/wall height/spray
  direction automatically).
- Contact-only fallback for rows where park geometry itself is unreliable.

Season split: 2021-2022 fit / 2023 model selection / 2024 final comparison (reusing
`mlb_luck_score.config.CALIBRATION_BASE_TRAIN_SEASONS`/`CALIBRATION_FIT_SEASONS`/
`CALIBRATION_EVAL_SEASONS` -- the SAME split already used for post-hoc calibration).
2025 remains untouched.

In [12]:
from mlb_luck_score.models.outfield_gating import assign_outfield_opportunity_gate

gate = None
if opp_df is not None:
    eligible = opp_df[opp_df["outfield_opportunity_eligible"]]
    gate = assign_outfield_opportunity_gate(eligible)
    print(gate.value_counts())

open_field               182846
near_wall                 31505
contact_only_fallback     12983
Name: count, dtype: int64


### 12.1 Model selection (fit 2021-2022, select on 2023)

In [13]:
from mlb_luck_score.models.compare_near_wall_models import (
    get_near_wall_rows,
    run_near_wall_model_selection,
)

near_wall_df = None
winner_variant = None
selection_metrics = None
trained_by_candidate = None
if df is not None:
    near_wall_df = get_near_wall_rows(df)
    print(f"{len(near_wall_df):,} near-wall-gated rows")
    winner_variant, selection_metrics, trained_by_candidate = run_near_wall_model_selection(near_wall_df)
    print(json.dumps(selection_metrics, indent=2))
    print("winner:", winner_variant)

31,505 near-wall-gated rows


{
  "near_wall_logistic_v07c": {
    "binary_log_loss": 0.4062367748889954,
    "expected_calibration_error": 0.016694629891917924
  },
  "near_wall_hgb_v07c": {
    "binary_log_loss": 0.3692767141712639,
    "expected_calibration_error": 0.017664792454005134
  }
}
winner: near_wall_hgb_v07c


### 12.2 Final comparison (2024): near-wall specialist vs. `open_field_v07`

In [14]:
from mlb_luck_score.config import CALIBRATION_EVAL_SEASONS, TRAIN_SEASONS
from mlb_luck_score.models.compare_near_wall_models import run_near_wall_final_comparison
from mlb_luck_score.models.compare_opportunity_models import _prepare_opportunity_columns
from mlb_luck_score.models.compare_park_aware import _prepare_venue_column
from mlb_luck_score.models.train_opportunity_model import train_opportunity_model

near_wall_comparison = None
open_field_trained_v07c = None
winner_trained = None
if near_wall_df is not None:
    prepared = _prepare_opportunity_columns(df)
    prepared = _prepare_venue_column(prepared) if "venue_id" in prepared.columns else prepared
    eligible_full = prepared[prepared["outfield_opportunity_eligible"].astype(bool)]
    open_field_train_rows = eligible_full[eligible_full["season"].isin(TRAIN_SEASONS)]
    open_field_trained_v07c = train_opportunity_model(open_field_train_rows, class_weight=None)

    winner_trained = trained_by_candidate[winner_variant]
    near_wall_comparison = run_near_wall_final_comparison(
        near_wall_df, open_field_trained_v07c, winner_variant, winner_trained
    )
    for variant, summary in near_wall_comparison.items():
        print(f"[{variant}] log_loss={summary['binary_log_loss']:.6f} "
              f"ece={summary['expected_calibration_error']:.6f} n={summary['sample_count']:,}")

[measured_contact_only_v07] log_loss=0.670934 ece=0.180439 n=7,853
[near_wall_v07c_candidate] log_loss=0.394988 ece=0.027564 n=7,853


### 12.3 Required-subgroup calibration: wall bands, wall height, spray sector, opportunity time

In [15]:
if near_wall_comparison is not None:
    rows = []
    for label, sub in near_wall_comparison["near_wall_v07c_candidate"]["near_wall_subgroups"].items():
        rows.append({"subgroup": label, "n": sub["sample_count"], "ece": sub["ece"]})
    print(pd.DataFrame(rows).set_index("subgroup"))

                                 n       ece
subgroup                                    
near_wall_5ft                 1947  0.039060
near_wall_10ft                3983  0.030853
near_wall_20ft                7853  0.027564
wall_height_short              366  0.052954
wall_height_medium             330  0.074802
wall_height_tall               348  0.113173
spray_sector_left             1388  0.051353
spray_sector_left_center      1612  0.057079
spray_sector_center           1687  0.024389
spray_sector_right_center     1681  0.027043
spray_sector_right            1485  0.028325
opportunity_time_q1_shortest  1965  0.043402
opportunity_time_q2           1965  0.031433
opportunity_time_q3           1961  0.030886
opportunity_time_q4_longest   1962  0.024130


### 12.4 Game-level paired bootstrap: near-wall specialist vs. `open_field_v07`

In [16]:
from mlb_luck_score.features.build_contact_features import OPPORTUNITY_TARGET_COLUMN
from mlb_luck_score.models.compare_near_wall_models import compute_near_wall_paired_bootstrap
from mlb_luck_score.models.train_opportunity_model import predict_opportunity_proba

near_wall_bootstrap = None
near_wall_final_df = None
if near_wall_comparison is not None:
    near_wall_final_df = near_wall_df[near_wall_df["season"].isin(CALIBRATION_EVAL_SEASONS)]
    y_true_nw = near_wall_final_df[OPPORTUNITY_TARGET_COLUMN].astype(int).to_numpy()
    open_field_cols = open_field_trained_v07c.numeric_features + open_field_trained_v07c.categorical_features
    winner_cols = winner_trained.numeric_features + winner_trained.categorical_features
    baseline_p_out = predict_opportunity_proba(open_field_trained_v07c, near_wall_final_df[open_field_cols])
    candidate_p_out = predict_opportunity_proba(winner_trained, near_wall_final_df[winner_cols])
    near_wall_bootstrap = compute_near_wall_paired_bootstrap(
        y_true_nw, baseline_p_out, candidate_p_out, near_wall_final_df["game_pk"]
    )
    print(json.dumps(near_wall_bootstrap, indent=2))

{
  "log_loss_delta": {
    "point_estimate": -0.27594572447120636,
    "ci_low": -0.2920072494759918,
    "ci_high": -0.25997800444870234,
    "n_reps": 500,
    "seed": 42,
    "resampling_unit": "game_pk"
  },
  "ece_delta": {
    "point_estimate": -0.15287451029074028,
    "ci_low": -0.16681748205945138,
    "ci_high": -0.13573894376061557,
    "n_reps": 500,
    "seed": 42,
    "resampling_unit": "game_pk"
  }
}


### 12.5 Controlled-perturbation directional checks

Both checks recompute their DEPENDENT derived features after overriding the raw one
(`_override_wall_distance_and_recompute` / `_override_launch_angle_and_recompute_hang_time`)
-- an earlier version of this check overrode `wall_distance_in_spray_direction` without
recomputing `projected_distance_to_wall_margin`/`absolute_distance_to_wall`, feeding the
model an internally-inconsistent row and producing a spurious backwards result. That is
fixed here; see the module docstring for the full account.

In [17]:
from mlb_luck_score.models.compare_near_wall_models import run_near_wall_perturbation_checks

near_wall_perturbation = None
if near_wall_comparison is not None:
    near_wall_perturbation = run_near_wall_perturbation_checks(winner_trained, near_wall_final_df)
    for name, r in near_wall_perturbation.items():
        print(f"{name}: low={r.mean_prob_low:.4f} high={r.mean_prob_high:.4f} delta={r.delta:+.4f} "
              f"expect_high_greater={r.expect_high_greater} passed={r.passed}")

wall_distance_direction: low=0.2557 high=0.5869 delta=+0.3312 expect_high_greater=True passed=True
hang_time_direction: low=0.2704 high=0.6634 delta=+0.3930 expect_high_greater=False passed=False


### Interpretation

The wall-distance check passes cleanly: a farther wall predicts materially HIGHER P(out)
than a close wall, as physically expected (a close wall lets the same batted ball clear
the fence or bang off it for extra bases; a far wall means the same ball is comparatively
more catchable).

The launch-angle/hang-time check is genuinely backwards from the open-field intuition,
even after the recompute fix. This may be a real, DIFFERENT physical relationship specific
to the near-wall CONDITIONAL population: among plays that all travel roughly the same
(long) distance to reach the wall by construction of the gate, achieving that distance via
a LOW launch angle requires much higher exit velocity (a flat, hard-hit line-drive
double/triple that gives the fielder little time to react) than achieving it via a HIGH
launch angle (a softer, higher-arcing fly ball the fielder has more time to track) -- the
reverse of the open-field-wide correlation, where distance itself varies freely. Per
CLAUDE.md's confounding-by-indication guidance, this is treated as an unresolved,
documented finding to investigate further, not a check to relax to force a pass.

### 12.6 No-open-field-regression check and validation summary

In [18]:
from mlb_luck_score.models.compare_near_wall_models import (
    check_no_open_field_regression,
    summarize_near_wall_validation,
)

near_wall_validation_summary = None
if near_wall_comparison is not None:
    regression_check = check_no_open_field_regression(open_field_trained_v07c, df)
    print("open-field regression check:", json.dumps(regression_check, indent=2))
    print()
    near_wall_validation_summary = summarize_near_wall_validation(
        near_wall_comparison, near_wall_bootstrap, near_wall_perturbation, regression_check
    )
    print(json.dumps(
        {k: v for k, v in near_wall_validation_summary.items() if k != "perturbation_detail"},
        indent=2, default=str,
    ))

open-field regression check: {
  "predictions_identical_via_gate": true,
  "gated_subset_sample_count": 46629,
  "gated_subset_binary_log_loss": 0.319420068883948,
  "gated_subset_ece": 0.06300260760968518,
  "ece_within_threshold": false
}

{
  "improves_log_loss_vs_open_field": true,
  "log_loss_delta": -0.27594572447120636,
  "bootstrap_supports_improvement": true,
  "wall_band_ece_issues": [],
  "other_subgroup_ece_issues": [
    "wall_height_short",
    "wall_height_medium",
    "wall_height_tall",
    "spray_sector_left",
    "spray_sector_left_center"
  ],
  "venue_ece_issues": [
    "14",
    "2",
    "3312",
    "2889",
    "15",
    "31",
    "12",
    "19",
    "5325",
    "2392",
    "22",
    "3",
    "2602",
    "4169",
    "4",
    "3289",
    "2681",
    "7",
    "1",
    "3313",
    "4705",
    "32",
    "2680",
    "2394",
    "17",
    "2395",
    "10",
    "680"
  ],
  "perturbation_checks_passed": false,
  "perturbation_failures": [
    "hang_time_direction"
  ],
 

### 12.7 Gated report: `execution_availability_status` per play

Per the task's reporting rule -- "Outfield execution score available for calibrated
open-field opportunities; provisional or unavailable for wall-adjacent opportunities" --
`near_wall_specialist_calibrated=False` above means near-wall rows are scored (never
silently blanked) but explicitly labeled `provisional_near_wall`, not equivalent-confidence
to open-field scores.

In [19]:
from mlb_luck_score.scoring.gated_outfield_report import build_gated_outfield_report

gated_report = None
if near_wall_validation_summary is not None:
    val_gated_df = eligible_full[eligible_full["season"].isin(CALIBRATION_EVAL_SEASONS)]
    gated_report = build_gated_outfield_report(
        val_gated_df,
        contact_trained,
        open_field_trained_v07c,
        winner_trained,
        near_wall_specialist_calibrated=near_wall_validation_summary["near_wall_specialist_calibrated"],
    )
    print(gated_report["execution_availability_status"].value_counts())
    print()
    print(gated_report.groupby("execution_availability_status")[["p_out_opportunity", "defensive_execution"]].mean())

execution_availability_status
available_open_field             46629
provisional_near_wall             7853
unavailable_geometry_fallback     3116
Name: count, dtype: int64

                               p_out_opportunity  defensive_execution
execution_availability_status                                        
available_open_field                    0.514234             0.050651
provisional_near_wall                   0.482567             0.027429
unavailable_geometry_fallback                NaN                  NaN


## 13. Explicit limitations

- **`measured_contact_only_v07` is the only implemented candidate.** No assumed defender
  starting position is used anywhere in this model.
- **`typical_position_proxy_v07` was investigated and not built** -- the only citable
  public starting-depth figures found (MLB.com/Statcast, 2015-2016) are stale relative to
  this project's 2021-2024 window, with documented evidence of a directional trend toward
  deeper outfield positioning since then. See `mlb_luck_score.models.
  compare_opportunity_models` module docstring for the sources checked.
- **Estimated hang time and landing location have no public ground truth to validate
  against** -- they are physics-motivated, unvalidated approximations (vacuum projectile
  motion, no drag model). Treat them as directional signals, never measured quantities.
- **Forward/lateral/backward defender movement is NOT evaluated** -- it requires a real
  starting position, which does not exist in public data. This is a documented gap, not a
  silently-dropped requirement.
- **Calibration is materially worse for near-wall plays and mid-range opportunity times**
  -- exactly where judging defense is hardest. Treat v0.7B execution values for those
  specific plays with extra caution.
- **v0.7B execution is opportunity-relative, not a measure of HOW a catch was made** -- no
  reaction time, route efficiency, or throwing accuracy is captured; only whether the
  opportunity was converted, relative to an average outfielder's expected rate.
- **The four components in Section 10 are reported separately, never combined** into one
  score, per the task's explicit instruction.
- **Version 0.7C's near-wall specialist is NOT yet marked calibrated** -- it shows a large, real, bootstrap-confirmed improvement over `open_field_v07` on wall-adjacent plays, but an unresolved launch-angle/hang-time perturbation-check finding (Section 12.5) and several subgroup/venue ECE issues (plausibly small-near-wall-sample noise against a threshold tuned for a much larger population, but not yet confirmed) keep it provisional. Per the task's reporting rule, near-wall execution scores are labeled `provisional_near_wall`, not `available_near_wall_calibrated`.\n- This notebook uses only 2021-2024 data. 2025 remains untouched.